In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests", "psycopg2-binary", "kafka-python-ng"])
print("requests + psycopg2 + kafka-python-ng ready")

requests + psycopg2 + kafka-python-ng ready


Initialize Spark with Iceberg + MinIO + REST catalog

In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *
import os, time, json, requests

S3_ENDPOINT = "http://minio:9000"
S3_BUCKET   = "s3a://warehouse"
BOOTSTRAP   = "kafka:9092"

spark = (
    SparkSession.builder
    .appName("CDC-Bronze")
    .config("spark.sql.catalog.lakehouse.s3.connection.ssl.enabled", "false")
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "rest")
    .config("spark.sql.catalog.lakehouse.uri", "http://iceberg-rest:8181")   
    .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint", S3_ENDPOINT)
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.defaultCatalog", "lakehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version}")
print(f"Kafka: {BOOTSTRAP}")
print(f"S3:    {S3_ENDPOINT}")
print("Ready!")

Spark 4.1.0
Kafka: kafka:9092
S3:    http://minio:9000
Ready!


In [5]:
!pip install python-dotenv

In [6]:
import os
import psycopg2
from dotenv import load_dotenv

# 1. Load variables from your .env file
load_dotenv() 

# 2. Get variables with Python-style defaults
# Syntax: os.getenv("KEY", "default_value")
DB_USER = os.getenv("PG_USER", "cdc_user")
DB_PASS = os.getenv("PG_PASSWORD", "cdc_pass")

PG_CONN = {
    "host": "postgres", 
    "port": 5432, 
    "dbname": "sourcedb", 
    "user": DB_USER, 
    "password": DB_PASS
}

def pg_execute(sql, fetch=False):
    # Use the dictionary unpacking as before
    conn = psycopg2.connect(**PG_CONN)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(sql)
    result = cur.fetchall() if fetch else None
    cur.close()
    conn.close()
    return result

# Now this should authenticate correctly
ver = pg_execute("SELECT version();", fetch=True)
print(f"PostgreSQL: {ver[0][0][:60]}...")

PostgreSQL: PostgreSQL 16.13 on x86_64-pc-linux-musl, compiled by gcc (A...


# 1. Debizium CDC Connector
• Register a Debezium PostgreSQL connector via the Connect REST API.

• Capture changes from public.customers and public.drivers using log-based CDC (WAL).

• Verify initial snapshot: Debezium emits op="r" (read) events for existing rows.

• Verify live changes: INSERT, UPDATE, DELETE appear in Kafka topics dbserver1.public.*.

In [7]:
CONNECT_URL = "http://connect:8083"

for i in range(30):
    try:
        r = requests.get(f"{CONNECT_URL}/")
        if r.status_code == 200:
            print(f"Kafka Connect is ready: {r.json()}")
            break
    except:
        pass
    print(f"Waiting for Kafka Connect... ({i+1})")
    time.sleep(3)
else:
    raise RuntimeError("Kafka Connect did not start in time!")

Kafka Connect is ready: {'version': '3.9.0', 'commit': 'a60e31147e6b01ee', 'kafka_cluster_id': 'MkU3OEVBNTcwNTJENDM2Qg'}


In [8]:
requests.delete(f"{CONNECT_URL}/connectors/pg-cdc-connector")
time.sleep(2)

connector_config = {
    "name": "pg-cdc-connector",
    "config": {
        "connector.class": "io.debezium.connector.postgresql.PostgresConnector",
        "database.hostname": "postgres",
        "database.port": "5432",
        "database.user": DB_USER,
        "database.password": DB_PASS,
        "database.dbname": "sourcedb",
        "topic.prefix": "dbserver1",
        "table.include.list": "public.customers,public.drivers",
        "plugin.name": "pgoutput",
        "snapshot.mode": "initial",
        "key.converter.schemas.enable": "false",
        "value.converter.schemas.enable": "false",
    }
}

r = requests.post(
    f"{CONNECT_URL}/connectors",
    headers={"Content-Type": "application/json"},
    data=json.dumps(connector_config),
)
print(f"Status: {r.status_code}")
print(json.dumps(r.json(), indent=2))

Status: 201
{
  "name": "pg-cdc-connector",
  "config": {
    "connector.class": "io.debezium.connector.postgresql.PostgresConnector",
    "database.hostname": "postgres",
    "database.port": "5432",
    "database.user": "cdc_user",
    "database.password": "bdmgroupc",
    "database.dbname": "sourcedb",
    "topic.prefix": "dbserver1",
    "table.include.list": "public.customers,public.drivers",
    "plugin.name": "pgoutput",
    "snapshot.mode": "initial",
    "key.converter.schemas.enable": "false",
    "value.converter.schemas.enable": "false",
    "name": "pg-cdc-connector"
  },
  "tasks": [],
  "type": "source"
}


In [9]:
time.sleep(10)

r = requests.get(f"{CONNECT_URL}/connectors/pg-cdc-connector/status")
status = r.json()
print(f"Connector: {status['connector']['state']}")
for task in status.get('tasks', []):
    print(f"Task {task['id']}: {task['state']}")

assert status['connector']['state'] == 'RUNNING', f"Connector not running: {status}"

Connector: RUNNING
Task 0: RUNNING


Verify Live Changes

In [10]:
def parse_cdc_events(kafka_df):
    """Parse Debezium CDC events from a RAW Kafka DataFrame (must have all Kafka columns)."""
    raw = kafka_df.select(
        F.col("offset").alias("kafka_offset"),
        F.col("partition").alias("kafka_partition"),
        F.col("timestamp").alias("kafka_timestamp"),
        F.col("value").cast("string").alias("raw_value"),
    ).filter(F.col("raw_value").isNotNull())

    return raw.select(
        "kafka_offset", "kafka_partition", "kafka_timestamp",
        F.get_json_object("raw_value", "$.payload.op").alias("op"),
        F.get_json_object("raw_value", "$.payload.before.id").cast("int").alias("before_id"),
        F.get_json_object("raw_value", "$.payload.before.name").alias("before_name"),
        F.get_json_object("raw_value", "$.payload.before.email").alias("before_email"),
        F.get_json_object("raw_value", "$.payload.before.country").alias("before_country"),
        F.get_json_object("raw_value", "$.payload.after.id").cast("int").alias("after_id"),
        F.get_json_object("raw_value", "$.payload.after.name").alias("after_name"),
        F.get_json_object("raw_value", "$.payload.after.email").alias("after_email"),
        F.get_json_object("raw_value", "$.payload.after.country").alias("after_country"),
        F.get_json_object("raw_value", "$.payload.source.lsn").cast("long").alias("source_lsn"),
        F.get_json_object("raw_value", "$.payload.ts_ms").cast("long").alias("ts_ms"),
    )



In [11]:
# Add a new cell to test live CDC
# 1. Insert a new record
pg_execute("INSERT INTO public.customers (name, email, country) VALUES ('Test User', 'test@example.com', 'Estonia');")

# 2. Update a record
pg_execute("UPDATE public.customers SET country = 'Latvia' WHERE email = 'test@example.com';")

# 3. Delete a record
pg_execute("DELETE FROM public.customers WHERE email = 'test@example.com';")

print("Live changes sent to Postgres. Wait a few seconds for Kafka...")
time.sleep(5)

# 4. Verify in Kafka
live_test_df = spark.read.format("kafka") \
    .option("kafka.bootstrap.servers", BOOTSTRAP) \
    .option("subscribe", "dbserver1.public.customers") \
    .option("startingOffsets", "earliest") \
    .load()

# Look for 'c' (create), 'u' (update), and 'd' (delete) operations
parsed_live = parse_cdc_events(live_test_df)
parsed_live.filter("op IN ('c', 'u', 'd')").show(5, truncate=80)

Live changes sent to Postgres. Wait a few seconds for Kafka...
+------------+---------------+-----------------------+---+---------+-----------+------------+--------------+--------+------------+-----------------------+-------------+----------+-------------+
|kafka_offset|kafka_partition|        kafka_timestamp| op|before_id|before_name|before_email|before_country|after_id|  after_name|            after_email|after_country|source_lsn|        ts_ms|
+------------+---------------+-----------------------+---+---------+-----------+------------+--------------+--------+------------+-----------------------+-------------+----------+-------------+
|          28|              0|2026-05-02 10:12:26.306|  u|     NULL|       NULL|        NULL|          NULL|      12|   Chen Mets|    chen.mets@inbox.org|      Estonia|  26595432|1777716745963|
|          29|              0|2026-05-02 10:12:28.314|  c|     NULL|       NULL|        NULL|          NULL|      35|Maria Muller| maria.muller@inbox.org|    Lit

In [12]:
from kafka.admin import KafkaAdminClient

admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP)
topics = admin.list_topics()
admin.close()

print("Kafka topics:")
for t in sorted(topics):
    print(f"  {t}")

assert "dbserver1.public.customers" in topics, "CDC topic not found! Check connector status."
print("\n✓ CDC topic 'dbserver1.public.customers' exists!")

Kafka topics:
  __consumer_offsets
  _connect_configs
  _connect_offsets
  _connect_statuses
  dbserver1.public.customers
  dbserver1.public.drivers
  taxi-trips

✓ CDC topic 'dbserver1.public.customers' exists!


# 2. Bronze CDC Layer (raw CDC event log)
• Read from the Kafka CDC topics using Spark (batch is fine).

• Parse the Debezium envelope: extract op, before.\*, after.\*, source.lsn, ts_ms from $.payload.*.

• Write every event **append-only** to a Bronze Iceberg table — never UPDATE or DELETE rows in Bronze.

• Include Kafka metadata (offset, partition, timestamp) alongside CDC fields.

• Handle tombstone records (null-value messages emitted after deletes for log compaction).

In [13]:
# Read CDC events as a batch for inspection
cdc_raw = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
        # taxi-trips
        # taxi-trips-january
        # dbserver1.public.customers
    .option("subscribe", "dbserver1.public.customers")
    .option("startingOffsets", "earliest")
    .load()
    .select(
        F.col("key").cast("string").alias("key"),
        F.col("value").cast("string").alias("value"),
        "offset",
        "timestamp",
    )
)

print(f"CDC events in topic: {cdc_raw.count()}")
cdc_raw.show(5, truncate=80)

CDC events in topic: 180
+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+------+-----------------------+
|                                                                             key|                                                                           value|offset|              timestamp|
+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+------+-----------------------+
|{"schema":{"type":"struct","fields":[{"type":"int32","optional":false,"defaul...|{"schema":{"type":"struct","fields":[{"type":"struct","fields":[{"type":"int3...|     0|2026-05-02 10:12:26.172|
|{"schema":{"type":"struct","fields":[{"type":"int32","optional":false,"defaul...|{"schema":{"type":"struct","fields":[{"type":"struct","fields":[{"type":"int3...|     1|2026-05-02 10:12:26.173|


In [14]:
# Pretty-print the first event — note the schema+payload wrapper
first_raw = cdc_raw.filter(F.col("value").isNotNull()).collect()[0]["value"]
first_event = json.loads(first_raw)
print("Top-level keys:", list(first_event.keys()))
print("\nPayload (the actual CDC data):")
print(json.dumps(first_event["payload"], indent=2))

Top-level keys: ['schema', 'payload']

Payload (the actual CDC data):
{
  "before": null,
  "after": {
    "id": 4,
    "name": "David Jonaitis",
    "email": "david@example.com",
    "country": "Lithuania",
    "created_at": 1777716626710078
  },
  "source": {
    "version": "3.0.8.Final",
    "connector": "postgresql",
    "name": "dbserver1",
    "ts_ms": 1777716745554,
    "snapshot": "first",
    "db": "sourcedb",
    "sequence": "[null,\"26595432\"]",
    "ts_us": 1777716745554467,
    "ts_ns": 1777716745554467000,
    "schema": "public",
    "table": "customers",
    "txId": 832,
    "lsn": 26595432,
    "xmin": null
  },
  "transaction": null,
  "op": "r",
  "ts_ms": 1777716745773,
  "ts_us": 1777716745773538,
  "ts_ns": 1777716745773538992
}


In [15]:
# Read fresh from Kafka (NOT from cdc_raw which already dropped partition)
kafka_raw = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", "dbserver1.public.customers")
    .option("startingOffsets", "earliest")
    .load()
)

parsed = parse_cdc_events(kafka_raw)

print("Parsed CDC events (initial snapshot, op='r'):")
parsed.select("kafka_offset", "op", "after_id", "after_name", "after_email", "after_country", "ts_ms") \
      .orderBy("kafka_offset").show(truncate=False)

Parsed CDC events (initial snapshot, op='r'):
+------------+---+--------+----------------+-------------------------+-------------+-------------+
|kafka_offset|op |after_id|after_name      |after_email              |after_country|ts_ms        |
+------------+---+--------+----------------+-------------------------+-------------+-------------+
|0           |r  |4       |David Jonaitis  |david@example.com        |Lithuania    |1777716745773|
|1           |r  |10      |Javier Garcia   |javier@example.com       |Spain        |1777716745777|
|2           |r  |7       |Grace Kim       |updated_7_303@mail.com   |South Korea  |1777716745777|
|3           |r  |12      |Chen Mets       |chen.mets@inbox.org      |Spain        |1777716745777|
|4           |r  |14      |Mia Park        |mia.park@example.com     |Brazil       |1777716745778|
|5           |r  |16      |Diego Muller    |diego.muller@inbox.org   |Italy        |1777716745778|
|6           |r  |19      |Yuki Nakamura   |yuki.nakamura@examp

The Bronze layer stores **every CDC event exactly as received** in an Iceberg table on MinIO.

• Append-only — we never update or delete rows in Bronze

• Preserves the complete history of all changes

• Our safety net for replay and debugging

Create the namespace

In [16]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.cdc")

DataFrame[]

Read CDC events from Kafka

In [17]:
raw = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", "dbserver1.public.customers,dbserver1.public.drivers")
    .option("startingOffsets", "earliest")
    .load()
)


Parse the Debezium envelope

In [18]:
from pyspark.sql import functions as F

raw_filtered = raw.filter(F.col("value").isNotNull())
'''
bronze_df = raw_filtered.select(
    F.col("topic"),
    F.col("partition").alias("kafka_partition"),
    F.col("offset").alias("kafka_offset"),
    F.col("timestamp").alias("kafka_timestamp"),
    F.get_json_object(F.col("value").cast("string"), "$.payload.op").alias("op"),
    F.get_json_object(F.col("value").cast("string"), "$.payload.ts_ms").cast("long").alias("ts_ms"),
    F.get_json_object(F.col("value").cast("string"), "$.payload.after.id").cast("int").alias("after_id"),
    F.get_json_object(F.col("value").cast("string"), "$.payload.after.name").alias("after_name"),
    F.get_json_object(F.col("value").cast("string"), "$.payload.after.email").alias("after_email"),
    F.get_json_object(F.col("value").cast("string"), "$.payload.after.country").alias("after_country"),
    F.get_json_object(F.col("value").cast("string"), "$.payload.before.id").cast("int").alias("before_id"),
)'''

bronze_df = raw.filter(F.col("value").isNotNull()).select(
    F.col("topic"),
    F.col("partition").alias("kafka_partition"),
    F.col("offset").alias("kafka_offset"),
    F.col("timestamp").alias("kafka_timestamp"),
    F.get_json_object(F.col("value").cast("string"), "$.payload.op").alias("op"),
    F.get_json_object(F.col("value").cast("string"), "$.payload.ts_ms").cast("long").alias("ts_ms"),
    F.get_json_object(F.col("value").cast("string"), "$.payload.after.id").cast("int").alias("after_id"),
    F.get_json_object(F.col("value").cast("string"), "$.payload.after.name").alias("after_name"),
    F.get_json_object(F.col("value").cast("string"), "$.payload.after.email").alias("after_email"),
    F.get_json_object(F.col("value").cast("string"), "$.payload.after.country").alias("after_country"),
    F.get_json_object(F.col("value").cast("string"), "$.payload.before.id").cast("int").alias("before_id"),
)

In [19]:
#Can preview
bronze_df.show(5, truncate=False)


+------------------------+---------------+------------+-----------------------+---+-------------+--------+----------------+-----------+-------------+---------+
|topic                   |kafka_partition|kafka_offset|kafka_timestamp        |op |ts_ms        |after_id|after_name      |after_email|after_country|before_id|
+------------------------+---------------+------------+-----------------------+---+-------------+--------+----------------+-----------+-------------+---------+
|dbserver1.public.drivers|0              |0           |2026-05-02 10:12:26.301|r  |1777716745790|7       |Omar Ride       |NULL       |NULL         |NULL     |
|dbserver1.public.drivers|0              |1           |2026-05-02 10:12:26.302|r  |1777716745790|2       |Sarah Wheels    |NULL       |NULL         |NULL     |
|dbserver1.public.drivers|0              |2           |2026-05-02 10:12:26.302|r  |1777716745790|13      |Ivan Andersen   |NULL       |NULL         |NULL     |
|dbserver1.public.drivers|0             

Write to Bronze Iceberg table

In [20]:
# createTable if not exists, then always append
table_name = "lakehouse.cdc.bronze_events"
if not spark.catalog.tableExists(table_name):
    bronze_df.writeTo(table_name).create()
else:
    bronze_df.writeTo(table_name).append()


Verify the Bronze table - count rows and sample rows

In [21]:
spark.sql("SELECT count(*) FROM lakehouse.cdc.bronze_events").show()


+--------+
|count(1)|
+--------+
|     251|
+--------+



In [22]:
spark.sql("""
  SELECT op, after_id, after_name, after_email, ts_ms
  FROM lakehouse.cdc.bronze_events
  LIMIT 5
""").show(truncate=False)

+---+--------+----------------+-----------+-------------+
|op |after_id|after_name      |after_email|ts_ms        |
+---+--------+----------------+-----------+-------------+
|r  |7       |Omar Ride       |NULL       |1777716745790|
|r  |2       |Sarah Wheels    |NULL       |1777716745790|
|r  |13      |Ivan Andersen   |NULL       |1777716745790|
|r  |12      |Olivia Mets     |NULL       |1777716745790|
|r  |17      |Fatima Johansson|NULL       |1777716745791|
+---+--------+----------------+-----------+-------------+



# 3. Silver CDC Layer (current-state mirror)
• Deduplicate Bronze: keep only the latest event per primary key (ROW_NUMBER over ts_ms DESC).

• Apply MERGE INTO to the Silver Iceberg table.

• After MERGE, Silver must exactly mirror the current state of the PostgreSQL source.

• Document your MERGE logic and explain why re-running it is idempotent.

op = 'd'             → DELETE the row

op in ('c','u','r')  → UPDATE if matched, INSERT if not

In [23]:
spark.sql("""
  CREATE TABLE IF NOT EXISTS lakehouse.cdc.silver_customers (
    id INT, name STRING, email STRING, country STRING, last_updated_ms BIGINT
  ) USING iceberg
""")


spark.sql("""
  CREATE TABLE IF NOT EXISTS lakehouse.cdc.silver_drivers (
    id INT, name STRING, email STRING, last_updated_ms BIGINT
  ) USING iceberg
""")

DataFrame[]

In [24]:
from pyspark.sql.window import Window

In [25]:
# Use COALESCE because for deletes, after_id is null but before_id has the key

bronze_with_key = bronze_df.withColumn(
  "entity_id", F.coalesce(F.col("after_id"), F.col("before_id"))
)

In [28]:
w = Window.partitionBy("entity_id").orderBy(F.col("ts_ms").desc())

deduped = (
    bronze_with_key
    .filter(F.col("op").isNotNull())
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .drop("rn")
)

In [29]:
deduped.createOrReplaceTempView("cdc_batch")

In [33]:
spark.sql("""

  MERGE INTO lakehouse.cdc.silver_customers AS t
  USING cdc_batch AS s
  ON t.id = s.entity_id

  WHEN MATCHED AND s.op = 'd' THEN DELETE

  WHEN MATCHED AND s.op IN ('c','u','r') THEN UPDATE SET
    t.name = s.after_name, t.email = s.after_email,
    t.country = s.after_country, t.last_updated_ms = s.ts_ms

  WHEN NOT MATCHED AND s.op != 'd' THEN INSERT
    (id, name, email, country, last_updated_ms)
    VALUES (s.after_id, s.after_name, s.after_email, s.after_country, s.ts_ms)
""")

spark.sql("""

  MERGE INTO lakehouse.cdc.silver_drivers AS t
  USING cdc_batch AS s
  ON t.id = s.entity_id

  WHEN MATCHED AND s.op = 'd' THEN DELETE

  WHEN MATCHED AND s.op IN ('c','u','r') THEN UPDATE SET
    t.name = s.after_name, t.email = s.after_email,
    t.last_updated_ms = s.ts_ms

  WHEN NOT MATCHED AND s.op != 'd' THEN INSERT
    (id, name, email, last_updated_ms)
    VALUES (s.after_id, s.after_name, s.after_email, s.ts_ms)
""")

DataFrame[]

In [34]:
spark.sql("SELECT count(*) FROM lakehouse.cdc.silver_customers").show()
spark.sql("SELECT count(*) FROM lakehouse.cdc.silver_drivers").show()

spark.sql("SELECT * FROM lakehouse.cdc.silver_customers ORDER BY id LIMIT 5").show(truncate=False)
spark.sql("SELECT * FROM lakehouse.cdc.silver_drivers ORDER BY id LIMIT 5").show(truncate=False)

+--------+
|count(1)|
+--------+
|     153|
+--------+

+--------+
|count(1)|
+--------+
|     155|
+--------+

+---+--------------+-----------------------+---------+---------------+
|id |name          |email                  |country  |last_updated_ms|
+---+--------------+-----------------------+---------+---------------+
|4  |David Jonaitis|updated_4_721@mail.com |Lithuania|1777717041301  |
|8  |Priya Fast    |NULL                   |NULL     |1777717604646  |
|13 |Ivan Andersen |NULL                   |NULL     |1777717388812  |
|20 |Mia Mets      |updated_20_790@test.net|Latvia   |1777717533079  |
|25 |Mateo Muller  |NULL                   |NULL     |1777717412692  |
+---+--------------+-----------------------+---------+---------------+

+---+--------------+-----------------------+---------------+
|id |name          |email                  |last_updated_ms|
+---+--------------+-----------------------+---------------+
|4  |David Jonaitis|updated_4_721@mail.com |1777717041301  |
|8  

# 4. Airflow DAG (orchestration)
• Create a DAG that orchestrates both pipelines end-to-end on a schedule.

• Configure retries (≥ 1); downstream tasks must skip if connector_health fails.

• Justify your schedule_interval — what freshness SLA does it support?

• The DAG must be idempotent: re-running for the same interval produces the same state.

Suggested task structure:
connector_health  — HTTP sensor: Debezium connector is RUNNING

bronze_cdc        — load new CDC events from Kafka into Bronze

bronze_taxi       — load new taxi events from Kafka into Bronze

silver_cdc        — MERGE Bronze → Silver for CDC tables

silver_taxi       — clean and enrich taxi events into Silver

gold_taxi         — aggregate Silver taxi data into Gold

validate          — compare Silver CDC row counts against PostgreSQL

# 5. Taxi Pipeline — Silver & Gold (improved from Project 2)
• Silver: parse timestamps, cast numeric types, drop invalid trips, enrich with zone names.

• Gold: at least one aggregation (e.g., hourly trips and average fare by pickup zone).

• This pipeline is now triggered by Airflow, not run as a standalone streaming job.

• Apply at least one improvement over your Project 2 submission based on feedback.

In [18]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.taxi")

DataFrame[]

Create a task that reads the taxi parquet files from data/ into a *Bronze Iceberg table* (lakehouse.taxi.bronze_trips). This is essentially what you built in Project 2, wrapped as a DAG task. Add a synthetic trip_id column using monotonically_increasing_id() — the raw taxi data doesn't have one, and you'll need it for joins later.